# Module 3 — Phase 1（ERP-only）Agent Demo

本 Notebook 演示 Phase 1 Agent 的完整流程：

1. 读取客户已写好的 PRD（`artifacts/prd.md`）
2. 基于 PRD，按 **8 维度结构** 生成系统提示词（`artifacts/system_prompt.md`）
3. 评审系统提示词（`artifacts/system_prompt_review.md`）

### 当前架构要点

- Agent 直接使用 `create_phase1_agent()` 运行，无子进程 / 无 SIGALRM
- System prompt 来自 `AGENTS.md`（内含 8 维度生成规则）
- PRD 是**只读输入**，Agent 不会修改 `prd.md`
- 工作流跳过「澄清提问」环节，直接从 PRD 生成提示词
- 系统提示词必须使用 **ReAct 模式**（Thought → Action → Observation → Final Answer）
- 第 8 维度末尾必须包含 **Few-Shot 范例**

---


## 1. 环境准备

请选择 Kernel：`Python (deep_agent_0530)`（对应仓库 `.venv`）

In [31]:
from __future__ import annotations

from pathlib import Path

from dotenv import load_dotenv

MODULE_DIR = Path.cwd()
PROJECT_ROOT = MODULE_DIR.parents[1]

# 加载环境变量（API Key 等）。
load_dotenv(PROJECT_ROOT / ".env")

print("MODULE_DIR =", MODULE_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)

MODULE_DIR = /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-3
PROJECT_ROOT = /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530


## 2. 检查输入产物

确保客户已写好的 PRD 以及指标口径字典模板已就位。

In [25]:
artifacts_dir = MODULE_DIR / "artifacts"
prd_path = artifacts_dir / "prd.md"
metrics_dict_path = artifacts_dir / "ERP_指标口径字典模板.md"

for p in [prd_path, metrics_dict_path]:
    print(p.name, "=>", "OK" if p.exists() else "MISSING")

prd.md => OK
ERP_指标口径字典模板.md => OK


### 预览 PRD 结构

快速确认 PRD 包含 .docx 模板要求的 13 个章节。

In [32]:
if prd_path.exists():
    prd_text = prd_path.read_text(encoding="utf-8")
    required_prd_sections = [
        "## 1. 基本资讯",
        "## 2. 背景与问题",
        "## 3. 产品目标",
        "## 4. 使用者与场景",
        "## 5. Agent 定位",
        "## 6. 核心流程",
        "## 7. 功能需求",
        "## 8. 能力边界",
        "## 9. 资料与工具",
        "## 10. 失败与兜底",
        "## 11. 评估指标",
        "## 12. 验收标准",
        "## 13. 风险与依赖",
    ]
    missing = [s for s in required_prd_sections if s not in prd_text]
    print("PRD 13 章节检查:", "PASS" if not missing else "FAIL")
    if missing:
        for s in missing:
            print("  - 缺失:", s)
    else:
        print("所有 13 章节均存在。")

PRD 13 章节检查: PASS
所有 13 章节均存在。


## 2.5 上传 PRD

如果你有自定义的 PRD 文件，可以使用下面的上传按钮将其上传到 `artifacts/prd.md`。
上传后，后续单元格会自动使用你上传的 PRD 进行处理。

> **注意**：上传的文件会**覆盖**现有的 `artifacts/prd.md`。
> 支持的格式：Markdown（`.md`）文件。

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

artifacts_dir = Path.cwd() / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

upload_btn = widgets.FileUpload(
    accept=".md,.txt",
    multiple=False,
    description="📤 上传 PRD",
    button_style="primary",
    style={"button_color": "#1976D2"},
    layout=widgets.Layout(width="auto"),
)

output = widgets.Output()

def on_upload(change):
    with output:
        clear_output(wait=True)
        uploaded = change["new"]
        if not uploaded:
            return
        # FileUpload returns {filename: {content: bytes, ...}, ...}
        for filename, file_info in uploaded.items():
            content = file_info["content"]
            if isinstance(content, bytes):
                content = content.decode("utf-8")
            dest = artifacts_dir / "prd.md"
            dest.write_text(content, encoding="utf-8")
            print(f"✅ 已上传: {filename}")
            print(f"📁 保存至: {dest}")
            print(f"📏 文件大小: {len(content):,} 字符")

upload_btn.observe(on_upload, names="value")

display(upload_btn, output)

FileUpload(value=(), accept='.md,.txt', button_style='primary', description='📤 上传 PRD', layout=Layout(width='a…

Output()

## 3. 查看所有 Prompt（运行前打印）

运行 Agent 前，先打印所有输入的 Prompt：AGENTS.md、subagents.yaml、phase1_prompts.py、skills/ SKILL.md。

In [4]:
def print_section(title: str, content: str) -> None:
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80)
    print(content)


# 1) AGENTS.md — 主 Agent 系统提示词
agents_md_path = MODULE_DIR / "AGENTS.md"
if agents_md_path.exists():
    print_section("AGENTS.md (主 Agent 系统提示词)", agents_md_path.read_text(encoding="utf-8"))

# 2) subagents.yaml — 子 Agent 角色与 Prompt
subagents_path = MODULE_DIR / "subagents.yaml"
if subagents_path.exists():
    print_section("subagents.yaml (子 Agent 定义)", subagents_path.read_text(encoding="utf-8"))

# 3) phase1_prompts.py — 提示词常量模板
prompts_py_path = MODULE_DIR / "phase1_prompts.py"
if prompts_py_path.exists():
    print_section("phase1_prompts.py 关键常量", prompts_py_path.read_text(encoding="utf-8"))

# 4) skills/ 各 SKILL.md
skills_dir = MODULE_DIR / "skills"
if skills_dir.exists():
    for skill_dir in sorted(skills_dir.iterdir()):
        skill_md = skill_dir / "SKILL.md"
        if skill_md.exists():
            print_section(f"skills/{skill_dir.name}/SKILL.md", skill_md.read_text(encoding="utf-8"))


  AGENTS.md (主 Agent 系统提示词)
# Phase 1 产品设计 Agent

你是一个只负责 Phase 1 的产品设计 Agent。

你的目标是**读取客户已写好的 `artifacts/prd.md`**，并基于 PRD 内容直接生成：

1. 一份基于 PRD 的 `artifacts/system_prompt.md`
2. 一份系统提示词评审报告 `artifacts/system_prompt_review.md`

## 重要前提

- **`artifacts/prd.md` 是客户（业务分析师/产品经理）写好的输入文件**，不是由你生成的。
- prd.md 应已包含完整的业务描述、目标用户、场景、Agent 角色、边界条件等。
- 你的工作是**读取并理解这个 PRD**，然后直接生成系统提示词。

## Phase 1 工作边界

你只负责以下内容：

- 读取并理解 `artifacts/prd.md`
- 基于 PRD 内容生成系统提示词
- 对生成的系统提示词执行评审

你不负责以下内容：

- 修改或覆盖 `artifacts/prd.md`
- 向用户提问或要求补充信息
- 详细用户需求文档
- 技术架构设计
- 技术实现方案
- 项目排期与项目计划
- 代码生成

以上内容属于 Phase 2 或之后的范围。

## 总体原则

1. **以 PRD 为唯一事实来源**
   - 系统提示词必须基于 PRD 生成。
   - 不允许在系统提示词中加入 PRD 未明确说明的业务事实。

2. **PRD 是客户输入，不可修改**
   - `artifacts/prd.md` 是客户的输出，你只能读取，不能写入或覆盖。

3. **避免臆造**
   - 不允许自己补造业务目标、审批规则、风险边界或工具权限。

4. **先评审，再完成**
   - 系统提示词生成后必须执行一次质量评审。

## 输出文件约定

请将中间产物和最终产物写入以下路径：

- `artifacts/system_prompt.md`（主要输出）
- `artifacts/system_prompt_review.md`

**注意：`artifacts/prd.md` 是客户输入，你只能读取，不能写入或覆盖。**

## 推荐工

## 4. 运行 Phase 1 Agent（Streaming）

直接调用 `create_phase1_agent()` — 无子进程，无需额外超时设置。

使用 **Streaming 模式**，实时显示 Agent 当前在做什么：
- `🔄 [Node]` — 当前执行的节点
- `🔧 [Tool]` — 调用的工具（read_file / write_file / task 委托等）
- `📝 [SubAgent]` — 子 Agent 的委派
- `💬 [Message]` — 消息内容摘要

Agent 工作流：
1. 读取 `artifacts/prd.md`
2. 按 **8 维度结构** 生成系统提示词 → `artifacts/system_prompt.md`
3. 评审系统提示词 → `artifacts/system_prompt_review.md`

In [27]:
all_messages = []
last_msg_count = 0

for event in agent.stream(
    {"messages": [{"role": "user", "content": user_task}]},
    stream_mode="values",
):
    messages = event.get("messages", [])
    if not messages:
        continue

    # Detect new messages since last step
    new_msgs = messages[last_msg_count:]
    last_msg_count = len(messages)

    for msg in new_msgs:
        all_messages.append(msg)
        if not hasattr(msg, "type"):
            continue

        if msg.type == "ai":
            tool_calls = getattr(msg, "tool_calls", None)
            if tool_calls:
                for tc in tool_calls:
                    tool_name = tc.get("name", "unknown")
                    tool_args = tc.get("args", {})
                    if tool_name == "task":
                        subagent = tool_args.get("subagent_type", tool_args.get("name", "?"))
                        desc = tool_args.get("description", "")[:120]
                        print(f"  🔧 [Tool] task -> {subagent}", flush=True)
                        if desc:
                            print(f"         {desc}", flush=True)
                    elif tool_name == "read_file":
                        print(f"  📖 [Tool] read_file: {tool_args.get('file_path', '?')}", flush=True)
                    elif tool_name == "write_file":
                        path = tool_args.get("file_path", "?")
                        print(f"  ✏️  [Tool] write_file: {path}", flush=True)
                    else:
                        print(f"  🔧 [Tool] {tool_name}", flush=True)
            if msg.content:
                preview = msg.content[:160].replace("\n", " ")
                suffix = "..." if len(msg.content) > 160 else ""
                print(f"  💬 [AI] {preview}{suffix}", flush=True)

        elif msg.type == "tool":
            content = (msg.content or "")[:120]
            if getattr(msg, "name", None) == "task":
                print(f"  📝 [SubAgent] 返回", flush=True)
                if content:
                    print(f"     {content}", flush=True)
            elif content:
                print(f"  📊 [Tool Result] {content}", flush=True)

print("\n=== ✅ Agent 执行完成 ===\n", flush=True)

  📖 [Tool] read_file: artifacts/prd.md
  🔧 [Tool] ls
  💬 [AI] Let me start by reading the PRD file to understand its contents.
  📊 [Tool Result] Error: File '/artifacts/prd.md' not found
  🔧 [Tool] ls
  💬 [AI] `artifacts/` 目录是空的，且 `artifacts/prd.md` 文件不存在。让我检查一下项目根目录下是否有其他位置存放了 PRD 文件。
  📊 [Tool Result] ["/.file", "/.nofollow/", "/.resolve/", "/.vol/", "/Applications/", "/Library/", "/System/", "/Users/", "/Volumes/", "/b
  🔧 [Tool] glob
  🔧 [Tool] glob
  📊 [Tool Result] ["/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-3/__pycache__/prd_schema.cpython-313.pyc", "/Us
  📊 [Tool Result] ["/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-3/AGENTS.md", "/Users/weiping/dev/Learn/langcha
  📖 [Tool] read_file: /Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/notebooks/module-3/artifacts/prd.md
  💬 [AI] 找到了 PRD 文件。让我读取它。
  📊 [Tool Result]      1	# 产品需求文档 Product Requirements Document (PRD)
     2	
     3	## 1. 基本资讯
     4	
     5	- **专案名称**：ERP

## 5. Token Usage & Cost Report

Agent 执行完成后，统计所有 LLM 调用的 Token 消耗与预估费用。

> **说明**：
> - Token 数据来自每次 LLM 返回的 `usage_metadata`（由模型 Provider 返回）
> - 子 Agent（SubAgent）的内部调用由子 Agent 自行计费，框架不向上透传
> - 费用按 DeepSeek Chat 定价估算：输入 0.5/百万 tokens，输出 2/百万 tokens
> - 如需精确成本，请以模型 Provider 账单为准

In [28]:
import os

print("=" * 80)
print("  Token Usage & Cost Report")
print("=" * 80)

# --- 1. 配置定价（根据模型 Provider 不同） ---
model_provider = os.environ.get("PHASE1_MODEL_PROVIDER", "deepseek")
model_name = os.environ.get("PHASE1_MODEL_NAME", "deepseek-chat")

# 定价：元/百万 tokens（可根据实际价格调整）
PRICING = {
    "deepseek": {"input": 0.5, "output": 2.0},
    "openai":    {"input": 2.5, "output": 10.0},
    "anthropic": {"input": 3.0, "output": 15.0},
    "minimax":   {"input": 0.5, "output": 2.0},
}
pricing = PRICING.get(model_provider, PRICING["deepseek"])

print(f"Model Provider : {model_provider}")
print(f"Model Name     : {model_name}")
print(f'Input Price    : \xa5{pricing["input"]}/1M tokens')
print(f'Output Price   : \xa5{pricing["output"]}/1M tokens')
print()

# --- 2. 汇总 Token 用量 ---
# 用 msg id 去重（避免 streaming update 中同一消息重复统计）
seen_ids = set()
input_tokens = 0
output_tokens = 0
total_tokens = 0
call_count = 0

for msg in all_messages:
    msg_id = id(msg)
    if msg_id in seen_ids:
        continue
    seen_ids.add(msg_id)

    # AIMessage 有 usage_metadata
    um = getattr(msg, "usage_metadata", None)
    if um:
        inp = um.get("input_tokens", 0) or 0
        out = um.get("output_tokens", 0) or 0
        tot = um.get("total_tokens", 0) or 0
        input_tokens += inp
        output_tokens += out
        total_tokens += tot
        call_count += 1

# --- 3. 计算费用 ---
input_cost = (input_tokens / 1_000_000) * pricing["input"]
output_cost = (output_tokens / 1_000_000) * pricing["output"]
total_cost = input_cost + output_cost

# --- 4. 打印报表 ---
print('=' * 60)
print(f"  {'Item':<30} {'Count':>12}")
print('=' * 60)
print(f'  {"LLM Calls":<30} {call_count:>12,}')
print(f'  {"Input Tokens":<30} {input_tokens:>12,}')
print(f'  {"Output Tokens":<30} {output_tokens:>12,}')
print(f'  {"Total Tokens":<30} {total_tokens:>12,}')
print('-' * 60)
print(f'  {"Input Cost (\xa5)":<30} {input_cost:>12.6f}')
print(f'  {"Output Cost (\xa5)":<30} {output_cost:>12.6f}')
print(f'  {"Total Cost (\xa5)":<30} {total_cost:>12.6f}')
print('=' * 60)

if call_count == 0:
    print()
    print("WARNING: No usage_metadata detected. Possible causes:")
    print("  - Model provider did not return token stats")
    print("  - Model does not support token stats")
    print("  - Agent did not make any LLM calls")

  Token Usage & Cost Report
Model Provider : deepseek
Model Name     : deepseek-chat
Input Price    : ¥0.5/1M tokens
Output Price   : ¥2.0/1M tokens

  Item                                  Count
  LLM Calls                                12
  Input Tokens                        155,649
  Output Tokens                         4,635
  Total Tokens                        160,284
------------------------------------------------------------
  Input Cost (¥)                     0.077825
  Output Cost (¥)                    0.009270
  Total Cost (¥)                     0.087095


## 6. 校验输出产物

检查生成的 `system_prompt.md` 是否严格遵循 8 维度 + ReAct 结构。

In [ ]:
system_prompt_path = artifacts_dir / "system_prompt.md"
system_prompt_review_path = artifacts_dir / "system_prompt_review.md"

print("=== 产物完整性检查 ===")
for p in [system_prompt_path, system_prompt_review_path]:
    print(p.name, "=>", "OK" if p.exists() else "MISSING")

In [ ]:
if not system_prompt_path.exists():
    print("system_prompt.md 不存在，请先运行 Agent。")
else:
    prompt_text = system_prompt_path.read_text(encoding="utf-8")

    # 8 维度标题检查
    dimension_checks = [
        ("1. Identity", "身份维度"),
        ("2. Background", "背景维度"),
        ("3. Mission", "任务维度"),
        ("4. Target", "目标维度"),
        ("5. Memories", "记忆维度"),
        ("6. Skills", "技能维度"),
        ("7. Can't Do", "禁止 & 兜底维度"),
        ("8. Steps", "步骤维度"),
    ]

    print("=== 8 维度结构检查 ===")
    all_ok = True
    for keyword, label in dimension_checks:
        found = keyword.lower() in prompt_text.lower()
        print(f"  {label}: {'OK' if found else 'MISSING'}")
        if not found:
            all_ok = False

    # ReAct 关键词检查
    react_checks = ["Thought", "Action", "Observation", "Final Answer"]
    print("\n=== ReAct 模式检查 ===")
    for kw in react_checks:
        found = kw.lower() in prompt_text.lower()
        print(f"  {kw}: {'OK' if found else 'MISSING'}")
        if not found:
            all_ok = False

    # Few-Shot 检查
    few_shot_found = "few-shot" in prompt_text.lower() or "范例" in prompt_text
    print(f"\n=== Few-Shot 范例检查 ===")
    print(f"  Few-Shot / 范例: {'OK' if few_shot_found else 'MISSING'}")
    if not few_shot_found:
        all_ok = False

    # 指标口径字典引用检查
    dict_ref_ok = "ERP_指标口径字典模板.md" in prompt_text
    print(f"\n=== 口径字典引用检查 ===")
    print(f"  引用口径字典: {'OK' if dict_ref_ok else 'MISSING'}")

    print(f"\n>>> 总体结果: {'ALL PASS ✅' if all_ok else 'SOME CHECKS FAILED ❌'} <<<")

In [ ]:
if not system_prompt_review_path.exists():
    print("评审报告不存在。")
else:
    review_text = system_prompt_review_path.read_text(encoding="utf-8")
    verdict = "pass" if "pass" in review_text.lower().split("**") else "N/A"
    # 从评审报告中提取结论
    for line in review_text.splitlines():
        if "结论" in line or "verdict" in line.lower() or "Verdict" in line or "评审结论" in line:
            print("评审结论:", line.strip())
    # 找到 pass/revise/blocked 状态
    for status in ["pass", "revise", "blocked"]:
        if status in review_text.lower():
            print(f"状态: {status} {'✅' if status == 'pass' else '⚠️'}")
            break

## 7. 预览生成产物

快速查看生成的两个核心文件前 60 行。

In [ ]:
def preview_text(path: Path, max_lines: int = 60) -> None:
    print("\n" + "=" * 80)
    print(path.name)
    print("=" * 80)
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


for p in [system_prompt_path, system_prompt_review_path]:
    if p.exists():
        preview_text(p)

---

## 附录：当前 Phase 1 文件结构

| 文件 | 说明 |
|------|------|
| `AGENTS.md` | Agent 全局规则，含 8 维度生成要求 |
| `phase1_agent.py` | Agent 入口，提供 `create_phase1_agent()` + `main()` |
| `phase1_prompts.py` | 提示词模板（参考用，Agent 实际读 AGENTS.md） |
| `subagents.yaml` | 子 Agent 定义（clarifier / prompt-architect / reviewer） |
| `skills/` | 技能文件（prd-intake / system-prompt-generator / reviewer） |
| `artifacts/prd.md` | **客户输入** — PRD（只读，Agent 不修改） |
| `artifacts/system_prompt.md` | **输出** — 8 维度系统提示词 |
| `artifacts/system_prompt_review.md` | **输出** — 评审报告 |